In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/multilingual-speech-recognition/sample_submission.csv
/kaggle/input/competitions/multilingual-speech-recognition/train.csv
/kaggle/input/competitions/multilingual-speech-recognition/test.csv
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00000.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00019.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00088.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00071.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00084.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00001.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00031.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00094.wav
/kaggl

In [26]:
!pip install jiwer

In [27]:
import os
import pandas as pd
import numpy as np
import torch
import librosa

from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from sklearn.model_selection import train_test_split
from jiwer import wer

In [28]:
DATA_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"

train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_sub = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

train_audio_path = f"{DATA_PATH}/competition_data/train"
test_audio_path = f"{DATA_PATH}/competition_data/test"

train_df.head()

,id,audio,text
0,0,audio_00000.wav,you had quoted plutarch line.
1,1,audio_00001.wav,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2,2,audio_00002.wav,to do his phd in engineering about four years ...
3,3,audio_00003.wav,maybe he was not at home.
4,4,audio_00004.wav,BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...


In [29]:
train_df["audio_path"] = train_df["audio"].apply(lambda x: os.path.join(train_audio_path, x))
test_df["audio_path"] = test_df["audio"].apply(lambda x: os.path.join(test_audio_path, x))

In [30]:
import re

# Keep transcript formatting consistent before tokenization and WER scoring.
def normalize_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

train_df["text"] = train_df["text"].apply(normalize_text)

In [31]:
train_df["text"]

0                           you had quoted plutarch line.
1       மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2       to do his phd in engineering about four years ...
3                               maybe he was not at home.
4       but we didn't break his old window you know ex...
                              ...                        
1995    நிறைய வகையான போட்டி நடக்கும் அப்புறம் வந்து உர...
1996    exclaimed the other as though more than surpri...
1997    पाकिस्तान ने आईएमएफ़ से करीब आठ से बारह डॉलर ब...
1998                               इसराइल की यात्रा की थी
1999              like destroying clothes and belongings.
Name: text, Length: 2000, dtype: object

In [32]:
# Hold out a small validation split to track WER during training.
train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

In [ ]:
# Load each clip at 16 kHz.
def load_audio(path):
    audio, sr = librosa.load(path, sr=16000)
    return audio

In [34]:
# Convert DataFrames so we can use dataset.map() and the Hugging Face trainer.
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

In [35]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [36]:
# Turn each row into Whisper-ready audio features and token labels.
def prepare_dataset(batch):
    audio = load_audio(batch["audio_path"])
    
    # Extract the log-Mel features Whisper uses as input.
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    batch["input_features"] = inputs.input_features[0]
    
    # Tokenize the reference transcript so the model can learn from it.
    labels = processor.tokenizer(
        batch["text"],
        return_tensors="pt"
    ).input_ids
    
    batch["labels"] = labels[0]
    return batch

In [37]:
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [38]:
# Pad audio features and labels to the longest item in the batch.
def data_collator(features):
    input_features = [f["input_features"] for f in features]
    label_features = [f["labels"] for f in features]
    
    batch = processor.feature_extractor.pad(
        {"input_features": input_features}, return_tensors="pt"
    )
    
    labels_batch = processor.tokenizer.pad(
        {"input_ids": label_features}, return_tensors="pt"
    )
    
    # Ignore padded label positions when computing the loss.
    labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
    
    batch["labels"] = labels
    return batch

In [39]:
# Use Word Error Rate to measure how close predictions are to the reference text.
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Restore ignored positions before decoding the labels back to text.
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": wer(label_str, pred_str)}

In [40]:
# Keep the run short enough to fit typical Kaggle GPU limits.
training_args = Seq2SeqTrainingArguments( 
    output_dir="./whisper-small-ft",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    # num_train_epochs=3
    max_steps=500,  # Safe for Kaggle runtime limits.
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=25,
    predict_with_generate=True,
    fp16=True,
    save_total_limit=2,
    save_strategy="steps",
)


# Hand the model, datasets, and helper functions to the trainer.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [41]:
# Fine-tune Whisper on the prepared training split.
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
200,1.869914,0.453464,0.262819
400,0.419855,0.328449,0.272906


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=1.8851623935699462, metrics={'train_runtime': 3535.2865, 'train_samples_per_second': 2.263, 'train_steps_per_second': 0.141, 'total_flos': 2.29944846974976e+18, 'train_loss': 1.8851623935699462, 'epoch': 4.426666666666667})

In [42]:
# Check validation loss and WER after training.
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.3239748775959015,
 'eval_wer': 0.261137573550014,
 'eval_runtime': 207.565,
 'eval_samples_per_second': 0.964,
 'eval_steps_per_second': 0.12,
 'epoch': 4.426666666666667}

In [43]:
# Run inference on one audio file and decode the generated transcript.
def predict(audio_path):
    audio = load_audio(audio_path)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(inputs["input_features"])
    
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

In [44]:
# Generate one transcript for each test clip.
predictions = []

for path in test_df["audio_path"]:
    predictions.append(predict(path))

In [45]:
# Save predictions in the competition submission format.
sample_sub["text"] = predictions
sample_sub.to_csv("submission.csv", index=False)

In [46]:
print("Done")

Done
